In [ ]:
import geopandas as gpd

# 1. Read ground-true dataset
gt_path = "path_to_your_real_ground_truth.geojson"  # replace this by your own path
gdf = gpd.read_file(gt_path)

# 2. check & transfer project to match firerx_ml
target_crs = "EPSG:32616"
if gdf.crs != target_crs:
    gdf = gdf.to_crs(target_crs)

# 3. Label digital encoding
# assume you have an attribute name 'Crop_Type'，including the type like 'WCC', 'Wheat', 'Fallow'
# map those names to 'class_id'
label_mapping = {
    'Fallow': 0,
    'WCC': 1,
    'Wheat': 2
}

if 'Crop_Type' in gdf.columns:
    gdf['class_id'] = gdf['Crop_Type'].map(label_mapping)
else:
    print("⚠️ Confirm your type name & edite the 'Crop_Type' column")

# 4. Data clean：exclude the invaild polygon 
gdf = gdf.dropna(subset=['class_id'])

# 5.  only include necessary column to reduce the computing consumption
gdf = gdf[['field_id', 'class_id', 'geometry']]  # set field_id as the only identification

# 6. save as for firerx_ml to read the clean data  
cleaned_gt_path = "./data/cleaned_target_labels.geojson"
gdf.to_file(cleaned_gt_path, driver="GeoJSON")

print(f"✅ label prepared! Totally having {len(gdf)} vaild ploygons")
print(" The distribution like below：\n", gdf['class_id'].value_counts())